# 03 Modelos de Filtrado Colaborativo

Entrenamiento y evaluación con SVD (Surprise) y KNN.

In [ ]:
import pandas as pd
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
from pathlib import Path
base = Path(r'd:\GitHub\Ruta_aprendizaje_2024\03-aplicación_de_modelos\proyecto_3\data')
ratings = pd.read_csv(base / 'ratings.csv')
reader = Reader(rating_scale=(ratings['rating'].min(), ratings['rating'].max()))
data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

## SVD

In [ ]:
svd = SVD(random_state=42)
svd.fit(trainset)
pred_svd = svd.test(testset)
rmse_svd = accuracy.rmse(pred_svd, verbose=False)
rmse_svd

## KNN Básico (item-item)

In [ ]:
sim_options = {'name':'cosine','user_based':False}
knn = KNNBasic(sim_options=sim_options)
knn.fit(trainset)
pred_knn = knn.test(testset)
rmse_knn = accuracy.rmse(pred_knn, verbose=False)
rmse_knn

## Top-N recomendaciones para un usuario

In [ ]:
user_id = ratings['userId'].iloc[0]
movie_ids = ratings['movieId'].unique()
rated = set(ratings.loc[ratings['userId']==user_id,'movieId'])
candidates = [m for m in movie_ids if m not in rated]
scores = [(m, svd.predict(user_id, m).est) for m in candidates]
topn = sorted(scores, key=lambda x: x[1], reverse=True)[:10]
topn